# RQ3 — Meta-Learner Training & Evaluation

**Research Question**: Can a meta-learner predict the best clustering method from dataset properties alone, before any clustering is run?

The current run evaluates 94 datasets, 11 feature options, and both classification and regression formulations. Regression remains the primary framework: the model predicts the full six-method LSE vector and recommends the argmax.

## Current baselines

| Baseline | Top-1 Acc | E[LSE] |
|----------|-----------|--------|
| Always k-means | 0.298 | 0.640 |
| de Souto default-ranking (always GMM) | 0.223 | 0.664 |
| Oracle | 1.000 | 0.731 |

## Current best results

- Best classification result: **C2 + Logistic Regression**, Top-1 = 0.436
- Saved classifier: **RF on Option A**, Top-1 = 0.426
- Best regression architecture: **Architecture A, ExtraTrees**, MAE = 0.1279, SRC = 0.486, Top-1 = 0.457, E[LSE] = 0.682
- Best feature ablation by MAE: **A+C**, MAE = 0.1233
- Best feature ablation by Top-1: **A+B** and **A+C2**, Top-1 = 0.468

**Outputs**: `outputs/models/meta_clf_optA.pkl`, `outputs/models/meta_reg_optA.pkl`, and `outputs/figures/ablation_regression.png`


In [1]:
import os, sys, pickle, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.metrics import confusion_matrix, classification_report

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(ROOT, 'src'))

from meta_learner import (
    extract_Xy_clf, extract_Xy_reg,
    loo_classify, loo_regress, loo_regress_per_method,
    baseline_always, default_ranking_baseline, oracle_expected_lse, baseline_random,
    bootstrap_ci_diff, mcnemar_exact_test, wilcoxon_paired_errors,
    report_clf_results, report_reg_results,
    build_classifier_candidates, build_regressor_candidates,
    LSE_COLS, METHOD_NAMES,
)

META_DIR   = os.path.join(ROOT, 'data', 'meta_table')
MODELS_DIR = os.path.join(ROOT, 'outputs', 'models')
FIGS_DIR   = os.path.join(ROOT, 'outputs', 'figures')
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(FIGS_DIR,   exist_ok=True)

SEED = 42
np.random.seed(SEED)
print('Imports OK')

Imports OK


In [2]:
df_a    = pd.read_csv(os.path.join(META_DIR, 'meta_training_optA.csv'))
df_b    = pd.read_csv(os.path.join(META_DIR, 'meta_training_optB.csv'))
df_c    = pd.read_csv(os.path.join(META_DIR, 'meta_training_optC.csv'))
df_c2   = pd.read_csv(os.path.join(META_DIR, 'meta_training_optC2.csv'))
df_d    = pd.read_csv(os.path.join(META_DIR, 'meta_training_optD.csv'))
df_ab   = pd.read_csv(os.path.join(META_DIR, 'meta_training_optAB.csv'))
df_ac   = pd.read_csv(os.path.join(META_DIR, 'meta_training_optAC.csv'))
df_ac2  = pd.read_csv(os.path.join(META_DIR, 'meta_training_optAC2.csv'))
df_ad   = pd.read_csv(os.path.join(META_DIR, 'meta_training_optAD.csv'))
df_cd   = pd.read_csv(os.path.join(META_DIR, 'meta_training_optCD.csv'))
df_rand = pd.read_csv(os.path.join(META_DIR, 'meta_training_optRand.csv'))

for name, df in [('A', df_a), ('B', df_b), ('C', df_c), ('C2', df_c2), ('D', df_d),
                  ('AB', df_ab), ('AC', df_ac), ('AC2', df_ac2),
                  ('AD', df_ad), ('CD', df_cd), ('Rand', df_rand)]:
    df.dropna(subset=['best_method'], inplace=True)
    df.reset_index(drop=True, inplace=True)
    print(f'Option {name}: {df.shape}')

print(f'\nBest method distribution (Option A):')
print(df_a['best_method'].value_counts().to_string())

Option A: (94, 29)
Option B: (94, 17)
Option C: (94, 17)
Option C2: (94, 36)
Option D: (94, 28)
Option AB: (94, 37)
Option AC: (94, 37)
Option AC2: (94, 56)
Option AD: (94, 48)
Option CD: (94, 36)
Option Rand: (94, 17)

Best method distribution (Option A):
best_method
kmeans       28
gmm          21
agg          17
autoenc      13
dictlearn    10
dbscan        5


In [3]:
y_true = df_a['best_method'].values

baseline_kmeans = baseline_always('kmeans', y_true)
oracle_lse_val  = oracle_expected_lse(df_a)
always_kmeans_lse = df_a['LSE_kmeans'].mean()

de_souto = default_ranking_baseline(df_a)
random_baseline = baseline_random(df_a, seed=SEED)

print('=== Baselines ===')
print(f'  Random (uniform over 6 methods) accuracy={random_baseline["accuracy_mean"]:.3f} '
      f'95%CI={tuple(round(v, 3) for v in random_baseline["accuracy_ci"])}  '
      f'expected_LSE={random_baseline["expected_lse_mean"]:.3f} '
      f'95%CI={tuple(round(v, 3) for v in random_baseline["expected_lse_ci"])}')
print(f'  Always k-means          accuracy={baseline_kmeans:.3f}  expected_LSE={always_kmeans_lse:.3f}')
print(f'  de Souto default-ranking: best method = {de_souto["default_method"]}  '
      f'accuracy={de_souto["baseline_accuracy"]:.3f}  expected_LSE={de_souto["baseline_expected_lse"]:.3f}')
print(f'  Oracle                   accuracy=1.000  expected_LSE={oracle_lse_val:.3f}')
print()
print('  Average LSE per method (de Souto ranking source):')
for m, v in de_souto['avg_lse_vector'].items():
    print(f'    {m:10s}  {v:.4f}')

=== Baselines ===
  Random (uniform over 6 methods) accuracy=0.167 95%CI=(0.096, 0.245)  expected_LSE=0.609 95%CI=(0.583, 0.635)
  Always k-means          accuracy=0.298  expected_LSE=0.640
  de Souto default-ranking: best method = gmm  accuracy=0.223  expected_LSE=0.664
  Oracle                   accuracy=1.000  expected_LSE=0.731

  Average LSE per method (de Souto ranking source):
    kmeans      0.6397
    dbscan      0.5207
    agg         0.6435
    gmm         0.6638
    autoenc     0.6233
    dictlearn   0.5593


## kNN k-Sweep on Option A

The current Option A kNN sweep selects **k=5** with LOO-CV accuracy **0.415**.


In [4]:
X_a, y_a, feat_cols_a, ids_a = extract_Xy_clf(df_a)

k_candidates = [1, 3, 5, 7, 9, 11, 13]
k_results = {}
for k in k_candidates:
    pipe = build_classifier_candidates(best_k=k, random_state=SEED)['kNN']
    res  = loo_classify(pipe, X_a, y_a)
    k_results[k] = res['accuracy']
    print(f'  k={k:2d}  accuracy={res["accuracy"]:.3f}')

best_k = max(k_results, key=k_results.get)
print(f'\nBest k = {best_k}  (accuracy={k_results[best_k]:.3f})')

  k= 1  accuracy=0.394


  k= 3  accuracy=0.404


  k= 5  accuracy=0.415


  k= 7  accuracy=0.340


  k= 9  accuracy=0.351


  k=11  accuracy=0.372


  k=13  accuracy=0.319

Best k = 5  (accuracy=0.415)


## Classification LOO-CV — All Models × All Feature Options

For comparison to prior classification-based work. The best observed classifier is **C2 + Logistic Regression** at 0.436 Top-1 accuracy. The saved production classifier remains **RF on Option A** at 0.426 because it uses the main label-free hand-crafted feature set.


In [5]:
clf_candidates = build_classifier_candidates(best_k=best_k, random_state=SEED)

options_clf = {
    'A (hand-crafted)' : df_a,
    'B (autoencoder)'  : df_b,
    'C (dictlearn)'    : df_c,
    'C2 (dl-v2)'       : df_c2,
    'D (distance)'     : df_d,
    'A+B'              : df_ab,
    'A+C (main claim)' : df_ac,
    'A+C2'             : df_ac2,
    'A+D'              : df_ad,
    'C+D'              : df_cd,
    'Random (sanity)'  : df_rand,
}

clf_results = {}
for opt_name, df_opt in options_clf.items():
    X_opt, y_opt, _, _ = extract_Xy_clf(df_opt)
    for model_name, pipe in clf_candidates.items():
        key = f'{opt_name} / {model_name}'
        res = loo_classify(pipe, X_opt, y_opt)
        clf_results[key] = res
        report_clf_results(key, res)

print('\nDone.')

  A (hand-crafted) / kNN                         acc=0.415  dist={'gmm': 22, 'kmeans': 31, 'autoenc': 10, 'dictlearn': 6, 'agg': 19, 'dbscan': 6}


  A (hand-crafted) / LogReg                      acc=0.372  dist={'autoenc': 15, 'dictlearn': 13, 'gmm': 21, 'kmeans': 25, 'agg': 12, 'dbscan': 8}


  A (hand-crafted) / SVC-RBF                     acc=0.383  dist={'autoenc': 22, 'kmeans': 28, 'dictlearn': 19, 'gmm': 13, 'agg': 6, 'dbscan': 6}


  A (hand-crafted) / ExtraTrees                  acc=0.415  dist={'gmm': 22, 'agg': 17, 'kmeans': 25, 'autoenc': 14, 'dictlearn': 9, 'dbscan': 7}


  A (hand-crafted) / RF                          acc=0.426  dist={'gmm': 23, 'agg': 17, 'kmeans': 30, 'autoenc': 11, 'dictlearn': 8, 'dbscan': 5}


...

  Random (sanity) / LogReg                       acc=0.255  dist={'dictlearn': 15, 'gmm': 14, 'kmeans': 19, 'dbscan': 13, 'autoenc': 10, 'agg': 23}


  Random (sanity) / SVC-RBF                      acc=0.245  dist={'dictlearn': 16, 'autoenc': 19, 'kmeans': 19, 'dbscan': 12, 'gmm': 15, 'agg': 13}


  Random (sanity) / ExtraTrees                   acc=0.170  dist={'dictlearn': 11, 'autoenc': 9, 'agg': 20, 'gmm': 18, 'kmeans': 28, 'dbscan': 8}


  Random (sanity) / RF                           acc=0.191  dist={'dictlearn': 6, 'gmm': 24, 'agg': 17, 'autoenc': 8, 'kmeans': 35, 'dbscan': 4}


  Random (sanity) / MLP                          acc=0.255  dist={'dictlearn': 10, 'gmm': 18, 'kmeans': 31, 'agg': 18, 'dbscan': 5, 'autoenc': 12}

Done.


In [6]:
rows = []
for key, res in clf_results.items():
    opt, model = key.split(' / ')
    rows.append({'Option': opt, 'Model': model, 'Top-1 Acc': round(res['accuracy'], 3)})

rows.append({'Option': '-', 'Model': 'Random (uniform)',        'Top-1 Acc': round(random_baseline['accuracy_mean'], 3)})
rows.append({'Option': '-', 'Model': 'Always k-means',           'Top-1 Acc': round(baseline_kmeans, 3)})
rows.append({'Option': '-', 'Model': f'de Souto ({de_souto["default_method"]})', 'Top-1 Acc': round(de_souto['baseline_accuracy'], 3)})
rows.append({'Option': '-', 'Model': 'Oracle',                   'Top-1 Acc': 1.000})

summary_df = pd.DataFrame(rows)
print('=== Classification Results (sorted) ===')
print(summary_df.sort_values('Top-1 Acc', ascending=False).to_string(index=False))

=== Classification Results (sorted) ===
          Option            Model  Top-1 Acc
               -           Oracle      1.000
      C2 (dl-v2)           LogReg      0.436
            A+C2               RF      0.426
A+C (main claim)               RF      0.426
A (hand-crafted)               RF      0.426
A (hand-crafted)              kNN      0.415
A+C (main claim)       ExtraTrees      0.415
            A+C2       ExtraTrees      0.415
             A+B       ExtraTrees      0.415
A (hand-crafted)       ExtraTrees      0.415
A+C (main claim)              MLP      0.415
      C2 (dl-v2)          SVC-RBF      0.415
             A+B              MLP      0.415
A+C (main claim)          SVC-RBF      0.404
             A+B          SVC-RBF      0.404
    D (distance)               RF      0.404
      C2 (dl-v2)       ExtraTrees      0.404
            A+C2              kNN      0.404
             A+D          SVC-RBF      0.394
             A+D       ExtraTrees      0.394
             A+

## Regression LOO-CV — Architecture A (Multi-Output)

Primary evaluation. Predicts the full LSE vector; recommended method = argmax. In the current run, **ExtraTrees** is best on Option A with MAE 0.1279, SRC 0.486, Top-1 0.457, Top-2+tie 0.553, and expected LSE 0.682.


In [7]:
reg_candidates = build_regressor_candidates(best_k=best_k, random_state=SEED)
X_a_reg, Y_a_reg, _, _ = extract_Xy_reg(df_a)

print('=== Architecture A: Multi-Output Regression (Option A features) ===')
print(f'{"Model":12s}  {"MAE":>6s}  {"SRC":>6s}  {"Top-1":>6s}  {"Top-2+tie":>9s}  {"E[LSE]":>7s}  {"Regret":>7s}')
print('-' * 75)

reg_results_A = {}
for model_name, pipe in reg_candidates.items():
    res = loo_regress(pipe, X_a_reg, Y_a_reg, df_a)
    reg_results_A[model_name] = res
    regret = oracle_lse_val - res['expected_lse']
    print(f'{model_name:12s}  {res["mae_mean"]:6.4f}  {res["src"]:6.3f}  '
          f'{res["top1_accuracy"]:6.3f}  {res["top2_accuracy"]:9.3f}  {res["expected_lse"]:7.3f}  {regret:7.3f}')

print('-' * 75)
print(f'{"Random":12s}  {"":>6s}  {"":>6s}  {random_baseline["accuracy_mean"]:6.3f}  {"":>9s}  '
      f'{random_baseline["expected_lse_mean"]:7.3f}  {oracle_lse_val - random_baseline["expected_lse_mean"]:7.3f}')
print(f'{"Always k-means":12s}  {"":>6s}  {"":>6s}  {baseline_kmeans:6.3f}  {"":>9s}  '
      f'{always_kmeans_lse:7.3f}  {oracle_lse_val - always_kmeans_lse:7.3f}')
print(f'{"de Souto":12s}  {"":>6s}  {"":>6s}  {de_souto["baseline_accuracy"]:6.3f}  {"":>9s}  '
      f'{de_souto["baseline_expected_lse"]:7.3f}  {oracle_lse_val - de_souto["baseline_expected_lse"]:7.3f}')
print(f'{"Oracle":12s}  {"":>6s}  {"":>6s}  {1.000:6.3f}  {"":>9s}  {oracle_lse_val:7.3f}  {0.0:7.3f}')

=== Architecture A: Multi-Output Regression (Option A features) ===
Model            MAE     SRC   Top-1  Top-2+tie   E[LSE]   Regret
---------------------------------------------------------------------------


kNN           0.1734   0.402   0.319      0.532    0.653    0.077


RF            0.1290   0.456   0.404      0.574    0.672    0.059


ExtraTrees    0.1279   0.486   0.457      0.553    0.682    0.049


MLP           0.1846   0.295   0.245      0.436    0.657    0.074


SVR-RBF       0.1501   0.387   0.287      0.479    0.648    0.083
---------------------------------------------------------------------------
Random                         0.167               0.609    0.122
Always k-means                   0.298               0.640    0.091
de Souto                       0.223               0.664    0.067
Oracle                         1.000               0.731    0.000


## Regression LOO-CV — Architecture B (Per-Method, de Souto Style)

Six separate single-output regressors, one per method. In the current run, **ExtraTrees** is again strongest by MAE (0.1286), but Architecture A is slightly better (0.1279), so the saved regressor uses Architecture A.


In [8]:
print('=== Architecture B: Per-Method Regression (Option A features) ===')
print(f'{"Model":12s}  {"MAE":>6s}  {"SRC":>6s}  {"Top-1":>6s}  {"Top-2+tie":>9s}  {"E[LSE]":>7s}')
print('-' * 65)

reg_results_B = {}
for model_name, pipe in reg_candidates.items():
    res = loo_regress_per_method(pipe, X_a_reg, Y_a_reg, df_a)
    reg_results_B[model_name] = res
    print(f'{model_name:12s}  {res["mae_mean"]:6.4f}  {res["src"]:6.3f}  '
          f'{res["top1_accuracy"]:6.3f}  {res["top2_accuracy"]:9.3f}  {res["expected_lse"]:7.3f}')

print('-' * 65)
best_A_name = min(reg_results_A, key=lambda n: reg_results_A[n]['mae_mean'])
best_B_name = min(reg_results_B, key=lambda n: reg_results_B[n]['mae_mean'])
print(f'\nBest Architecture A: {best_A_name}  MAE={reg_results_A[best_A_name]["mae_mean"]:.4f}')
print(f'Best Architecture B: {best_B_name}  MAE={reg_results_B[best_B_name]["mae_mean"]:.4f}')
winner = 'A' if reg_results_A[best_A_name]['mae_mean'] <= reg_results_B[best_B_name]['mae_mean'] else 'B'
print(f'Winner: Architecture {winner}')

=== Architecture B: Per-Method Regression (Option A features) ===
Model            MAE     SRC   Top-1  Top-2+tie   E[LSE]
-----------------------------------------------------------------


kNN           0.1734   0.402   0.319      0.532    0.653


RF            0.1321   0.485   0.372      0.585    0.664


ExtraTrees    0.1286   0.443   0.415      0.596    0.673


MLP           0.1994   0.399   0.383      0.457    0.665


SVR-RBF       0.1501   0.387   0.287      0.479    0.648
-----------------------------------------------------------------

Best Architecture A: ExtraTrees  MAE=0.1279
Best Architecture B: ExtraTrees  MAE=0.1286
Winner: Architecture A


## Feature Representation Ablation

Compares all feature options using the best regression model, ExtraTrees. In the current run, **A+C** has the lowest MAE (0.1233), **A+B** has the highest SRC (0.511), and **A+B / A+C2** have the highest Top-1 (0.468). All learned structured options beat the random sanity baseline by MAE.


In [9]:
best_reg_model = min(reg_results_A, key=lambda n: reg_results_A[n]['mae_mean'])
best_reg_pipe  = reg_candidates[best_reg_model]

print(f'Using best model: {best_reg_model}')
print()
print(f'{"Option":20s}  {"MAE":>6s}  {"SRC":>6s}  {"Top-1":>6s}  {"Top-2+tie":>9s}  {"E[LSE]":>7s}  {"Regret":>7s}')
print('-' * 82)

abl_results = {}
for opt_name, df_opt in options_clf.items():
    X_opt, Y_opt, _, _ = extract_Xy_reg(df_opt)
    res = loo_regress(best_reg_pipe, X_opt, Y_opt, df_opt)
    abl_results[opt_name] = res
    regret = oracle_lse_val - res['expected_lse']
    print(f'{opt_name:20s}  {res["mae_mean"]:6.4f}  {res["src"]:6.3f}  '
          f'{res["top1_accuracy"]:6.3f}  {res["top2_accuracy"]:9.3f}  {res["expected_lse"]:7.3f}  {regret:7.3f}')

print('-' * 82)
print(f'{"Random baseline":20s}  {"":>6s}  {"":>6s}  {random_baseline["accuracy_mean"]:6.3f}  {"":>9s}  '
      f'{random_baseline["expected_lse_mean"]:7.3f}  {oracle_lse_val - random_baseline["expected_lse_mean"]:7.3f}')
print(f'{"de Souto baseline":20s}  {"":>6s}  {"":>6s}  '
      f'{de_souto["baseline_accuracy"]:6.3f}  {"":>9s}  {de_souto["baseline_expected_lse"]:7.3f}  '
      f'{oracle_lse_val - de_souto["baseline_expected_lse"]:7.3f}')
print(f'{"Oracle":20s}  {"":>6s}  {"":>6s}  {1.000:6.3f}  {"":>9s}  {oracle_lse_val:7.3f}  {0.0:7.3f}')

# Highlight the key comparisons
print('\n=== Key comparisons ===')
for pair in [('A (hand-crafted)', 'A+B'), ('A (hand-crafted)', 'A+C (main claim)'),
              ('A+C (main claim)', 'Random (sanity)')]:
    a_mae = abl_results[pair[0]]['mae_mean']
    b_mae = abl_results[pair[1]]['mae_mean']
    delta = a_mae - b_mae
    winner = pair[1] if delta > 0 else pair[0]
    print(f'  {pair[0]} vs {pair[1]}: ΔMAE={delta:+.4f} → {winner} wins')

Using best model: ExtraTrees

Option                   MAE     SRC   Top-1  Top-2+tie   E[LSE]   Regret
----------------------------------------------------------------------------------


A (hand-crafted)      0.1279   0.486   0.457      0.553    0.682    0.049


B (autoencoder)       0.1627   0.497   0.362      0.532    0.673    0.058


C (dictlearn)         0.1721   0.505   0.383      0.521    0.671    0.059


C2 (dl-v2)            0.1624   0.502   0.426      0.564    0.687    0.044


...

A+C (main claim)      0.1233   0.491   0.404      0.585    0.673    0.058


A+C2                  0.1290   0.496   0.468      0.617    0.681    0.050


A+D                   0.1268   0.477   0.394      0.543    0.677    0.054


C+D                   0.1568   0.501   0.404      0.596    0.677    0.054


Random (sanity)       0.1908   0.319   0.245      0.404    0.646    0.085
----------------------------------------------------------------------------------
Random baseline                        0.167               0.609    0.122
de Souto baseline                      0.223               0.664    0.067
Oracle                                 1.000               0.731    0.000

=== Key comparisons ===
  A (hand-crafted) vs A+B: ΔMAE=+0.0017 → A+B wins
  A (hand-crafted) vs A+C (main claim): ΔMAE=+0.0046 → A+C (main claim) wins
  A+C (main claim) vs Random (sanity): ΔMAE=-0.0675 → A+C (main claim) wins


In [10]:
opt_names  = list(abl_results.keys())
maes       = [abl_results[n]['mae_mean'] for n in opt_names]
src_vals   = [abl_results[n]['src'] for n in opt_names]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

colors = ['#4C72B0' if 'A' in n and '(' not in n.split('A')[1][:2] else
          '#55A868' if '+C' in n else
          '#DD8452' if '+B' in n else
          '#999999' for n in opt_names]

ax1.bar(opt_names, maes, color=colors)
ax1.set_ylabel('MAE (lower = better)')
ax1.set_title('Regression MAE by Feature Option')
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=30, ha='right')

ax2.bar(opt_names, src_vals, color=colors)
ax2.set_ylabel('Spearman Rank Corr (higher = better)')
ax2.set_title('Ranking Quality (SRC) by Feature Option')
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=30, ha='right')

plt.tight_layout()
path = os.path.join(FIGS_DIR, 'ablation_regression.png')
fig.savefig(path, dpi=130)
plt.close()
print(f'Saved → {path}')

Saved → C:\MLResearch\outputs\figures\ablation_regression.png


## Statistical Significance

Point estimates above don't say whether observed differences are larger than
LOO-CV noise on 94 datasets. Two paired tests, computed on the *same* LOO-CV
predictions already produced above (no refitting):

- **McNemar's exact test** + **paired bootstrap CI** on Top-1 correctness masks,
  for the headline classification comparisons (best overall vs. saved production
  model, and saved model vs. the baselines).
- **Wilcoxon signed-rank test** on per-dataset absolute LSE error, for the
  regression feature-ablation comparisons (A vs. A+C "main claim", and A+C vs.
  the random-feature sanity check).


In [11]:
print('=== Classification: paired tests on LOO-CV correctness masks ===\n')

mask_c2_logreg = clf_results['C2 (dl-v2) / LogReg']['correct_mask']
mask_rf_a      = clf_results['A (hand-crafted) / RF']['correct_mask']
mask_kmeans    = (y_true == 'kmeans')
mask_desouto   = (y_true == de_souto['default_method'])

clf_pairs = [
    ('C2+LogReg (best overall) vs RF+A (saved model)', mask_c2_logreg, mask_rf_a),
    ('RF+A (saved model) vs Always k-means',            mask_rf_a,     mask_kmeans),
    ('RF+A (saved model) vs de Souto default-ranking',  mask_rf_a,     mask_desouto),
]

for label, ma, mb in clf_pairs:
    mc = mcnemar_exact_test(ma, mb)
    ci = bootstrap_ci_diff(ma, mb, seed=SEED)
    print(f'{label}')
    print(f'  Δaccuracy = {ci["observed_diff"]:+.3f}   95% bootstrap CI = '
          f'[{ci["ci_lo"]:+.3f}, {ci["ci_hi"]:+.3f}]')
    print(f'  McNemar: n10(a-only-right)={mc["n10"]}  n01(b-only-right)={mc["n01"]}  '
          f'p={mc["p_value"]:.3f}')
    print()

print('=== Regression: Wilcoxon signed-rank on per-dataset |error| ===\n')

reg_pairs = [
    ('A (hand-crafted) vs A+C (main claim)', abl_results['A (hand-crafted)'], abl_results['A+C (main claim)']),
    ('A+C (main claim) vs Random (sanity)',   abl_results['A+C (main claim)'], abl_results['Random (sanity)']),
    ('A (hand-crafted) vs A+B',               abl_results['A (hand-crafted)'], abl_results['A+B']),
]

for label, res_a, res_b in reg_pairs:
    wt = wilcoxon_paired_errors(res_a['abs_err_per_dataset'], res_b['abs_err_per_dataset'])
    print(f'{label}')
    print(f'  n={wt["n"]}  mean(|err_a|-|err_b|)={wt["mean_diff"]:+.4f}  '
          f'Wilcoxon stat={wt["statistic"]:.1f}  p={wt["p_value"]:.3f}')
    print()

print('Note: p < 0.05 marks a difference unlikely under LOO-CV noise alone;')
print('it is not a claim about effect size, which the CI/mean_diff columns give.')

=== Classification: paired tests on LOO-CV correctness masks ===



C2+LogReg (best overall) vs RF+A (saved model)
  Δaccuracy = +0.011   95% bootstrap CI = [-0.085, +0.106]
  McNemar: n10(a-only-right)=11  n01(b-only-right)=10  p=1.000



RF+A (saved model) vs Always k-means
  Δaccuracy = +0.128   95% bootstrap CI = [+0.000, +0.255]
  McNemar: n10(a-only-right)=25  n01(b-only-right)=13  p=0.073



RF+A (saved model) vs de Souto default-ranking
  Δaccuracy = +0.202   95% bootstrap CI = [+0.074, +0.330]
  McNemar: n10(a-only-right)=29  n01(b-only-right)=10  p=0.003

=== Regression: Wilcoxon signed-rank on per-dataset |error| ===

A (hand-crafted) vs A+C (main claim)
  n=82  mean(|err_a|-|err_b|)=+0.0046  Wilcoxon stat=1332.0  p=0.088

A+C (main claim) vs Random (sanity)
  n=82  mean(|err_a|-|err_b|)=-0.0675  Wilcoxon stat=538.0  p=0.000

A (hand-crafted) vs A+B
  n=82  mean(|err_a|-|err_b|)=+0.0017  Wilcoxon stat=1548.0  p=0.478

Note: p < 0.05 marks a difference unlikely under LOO-CV noise alone;
it is not a claim about effect size, which the CI/mean_diff columns give.


## Save Best Models

The current saved models are:
- `meta_clf_optA.pkl`: RF classifier trained on Option A features
- `meta_reg_optA.pkl`: ExtraTrees multi-output regressor trained on Option A features

These are consumed by notebook 06 for SHAP analysis and notebook 07 for showcase evaluation.


In [12]:
# Best classification model (Option A)
best_clf_name = max(clf_candidates,
                    key=lambda n: clf_results[f'A (hand-crafted) / {n}']['accuracy'])
best_clf_pipe = clf_candidates[best_clf_name]
final_clf = clone(best_clf_pipe)
final_clf.fit(X_a, y_a)
clf_path = os.path.join(MODELS_DIR, 'meta_clf_optA.pkl')
with open(clf_path, 'wb') as f:
    pickle.dump({'pipeline': final_clf, 'feature_cols': feat_cols_a}, f)
print(f'Classifier saved → {clf_path}  ({best_clf_name})')

# Best regression model (Option A, Architecture A)
best_reg_name = min(reg_results_A, key=lambda n: reg_results_A[n]['mae_mean'])
best_reg_pipe = reg_candidates[best_reg_name]
final_reg = clone(best_reg_pipe)
X_a_r, Y_a_r, feat_cols_a_r, _ = extract_Xy_reg(df_a)
valid_mask = ~np.isnan(Y_a_r).any(axis=1)
final_reg.fit(X_a_r[valid_mask], Y_a_r[valid_mask])
reg_path = os.path.join(MODELS_DIR, 'meta_reg_optA.pkl')
with open(reg_path, 'wb') as f:
    pickle.dump({'pipeline': final_reg, 'feature_cols': feat_cols_a_r,
                 'lse_cols': LSE_COLS, 'method_names': METHOD_NAMES}, f)
print(f'Regressor  saved → {reg_path}  ({best_reg_name})')

print(f'\nPhase complete. Best classifier: {best_clf_name}, best regressor: {best_reg_name}')
print('Ready for 06_shap_analysis.ipynb')

Classifier saved → C:\MLResearch\outputs\models\meta_clf_optA.pkl  (RF)


Regressor  saved → C:\MLResearch\outputs\models\meta_reg_optA.pkl  (ExtraTrees)

Phase complete. Best classifier: RF, best regressor: ExtraTrees
Ready for 06_shap_analysis.ipynb


In [13]:
# Also save a C2 (dictionary-learning) + LogReg classifier bundle, for
# comparison against the deployed Option A classifier on the showcase set.
# Motivation: C2+LogReg has meaningfully better minority-class recall in
# training LOO-CV (autoenc 0.38 vs 0.31, dictlearn 0.40 vs 0.20, dbscan 0.80
# vs 0.60 for RF+A) despite statistically indistinguishable overall accuracy
# (McNemar p=1.000, see the significance-testing cell above) -- RF+A was
# selected as the deployed model purely by raw accuracy, a criterion blind to
# this imbalance. This bundle lets 07_showcase_eval.ipynb test both models on
# held-out data using a choice justified by training-set metrics, decided
# before looking at any showcase result.
X_c2, y_c2, feat_cols_c2, _ = extract_Xy_clf(df_c2)
final_clf_c2 = clone(clf_candidates['LogReg'])
final_clf_c2.fit(X_c2, y_c2)
clf_c2_path = os.path.join(MODELS_DIR, 'meta_clf_optC2.pkl')
with open(clf_c2_path, 'wb') as f:
    pickle.dump({'pipeline': final_clf_c2, 'feature_cols': feat_cols_c2}, f)
print(f'Classifier saved -> {clf_c2_path}  (LogReg on Option C2)')


Classifier saved -> C:\MLResearch\outputs\models\meta_clf_optC2.pkl  (LogReg on Option C2)
